# EDA — Passos Mágicos Datathon
**Objetivo:** Explorar os dados do PEDE (2020-2022) e do dataset 2024 para construir um modelo preditivo de risco de defasagem escolar.

**Conclusões-chave desta análise:**
- O target será `DEFASAGEM` binarizado: `1 = defasado (valor < 0)`, `0 = no nível ou adiantado`
- Feature mais correlacionada com defasagem: **IAN** (Indicador de Adequação ao Nível)
- Várias colunas do Excel sofreram corrupção de formato (datas onde deveriam ser floats) — tratadas no preprocessing
- O dataset 2024 é limpo e será usado como base de produção da API

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

# Caminhos dos dados
DATA_MAIN = '../data/PEDE_PASSOS_DATASET_FIAP.xlsx'
DATA_2024 = '../data/BASE DE DADOS PEDE 2024 - DATATHON.xlsx'

df_main = pd.read_excel(DATA_MAIN)
df_2024 = pd.read_excel(DATA_2024)

print(f'Dataset principal: {df_main.shape}')
print(f'Dataset 2024: {df_2024.shape}')

## 1. Visão Geral dos Datasets

In [ ]:
print('=== Dataset Principal — Primeiras linhas ===')
df_main.head(3)

In [ ]:
print('=== Dataset 2024 — Primeiras linhas ===')
df_2024.head(3)

In [ ]:
print('Tipos de dados — Dataset principal:')
df_main.dtypes

## 2. Análise do Target: DEFASAGEM

**Interpretação do valor:**
- Negativo (ex: -1, -2): aluno está abaixo do nível esperado → **defasado**
- Zero: aluno está no nível esperado
- Positivo: aluno está acima do nível esperado

**Decisão de modelagem:** binarizamos como `1 = defasado (< 0)`, `0 = no nível ou adiantado`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dataset principal - DEFASAGEM_2021
defas_counts = df_main['DEFASAGEM_2021'].value_counts(dropna=True).sort_index()
axes[0].bar(defas_counts.index.astype(str), defas_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuição DEFASAGEM_2021 (Dataset Principal)', fontsize=12)
axes[0].set_xlabel('Grau de Defasagem')
axes[0].set_ylabel('Quantidade de Alunos')
axes[0].axvline(x=1.5, color='red', linestyle='--', alpha=0.5, label='Corte (0)')

# Dataset 2024
defas_2024 = df_2024['Defas'].value_counts().sort_index()
axes[1].bar(defas_2024.index.astype(str), defas_2024.values, color='coral', edgecolor='white')
axes[1].set_title('Distribuição Defas (Dataset 2024)', fontsize=12)
axes[1].set_xlabel('Grau de Defasagem')
axes[1].set_ylabel('Quantidade de Alunos')

plt.tight_layout()
plt.show()

print('\n--- Dataset Principal ---')
print(df_main['DEFASAGEM_2021'].value_counts(dropna=False))
print('\n--- Dataset 2024 ---')
print(df_2024['Defas'].value_counts())

In [ ]:
# Target binário — proporção de defasados
df_main['TARGET'] = (df_main['DEFASAGEM_2021'] < 0).astype(int)
df_2024['TARGET'] = (df_2024['Defas'] < 0).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, df, title in zip(axes, [df_main, df_2024], ['Dataset Principal (2021)', 'Dataset 2024']):
    tgt = df['TARGET'].dropna()
    counts = tgt.value_counts()
    ax.pie(counts, labels=['Defasado', 'No Nível/Adiantado'], autopct='%1.1f%%',
           colors=['#e74c3c', '#2ecc71'], startangle=90)
    ax.set_title(f'Proporção de Defasagem — {title}')

plt.suptitle('Target Binário: 1 = Defasado, 0 = Não Defasado', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f'Dataset principal — defasados com DEFASAGEM_2021 disponível: {df_main["TARGET"].value_counts()[1]} ({df_main["TARGET"].value_counts(normalize=True)[1]*100:.1f}%)')
print(f'Dataset 2024 — defasados: {df_2024["TARGET"].value_counts()[1]} ({df_2024["TARGET"].value_counts(normalize=True)[1]*100:.1f}%)')

## 3. Análise de Valores Nulos

In [ ]:
nulls = df_main.isnull().sum().sort_values(ascending=False)
nulls_pct = (nulls / len(df_main) * 100).round(1)
null_df = pd.DataFrame({'Nulos': nulls, 'Percentual': nulls_pct}).query('Nulos > 0')

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(null_df.index, null_df['Percentual'], color='steelblue')
ax.axvline(x=50, color='red', linestyle='--', label='50% limite')
ax.set_xlabel('% de valores nulos')
ax.set_title('Completude das Colunas — Dataset Principal')
ax.legend()
plt.tight_layout()
plt.show()

print('\nObservação:')
print('  - 487 registros (36%) não têm dados de 2022 → alunos que saíram antes de 2022')
print('  - NOTA_ING_2022: 79% nulos → excluir desta feature set')
print('  - DEFASAGEM_2021: 49% nulos → alunos que entraram após 2021 (novos ingressantes)')

## 4. Problema com Colunas do Excel (Artefatos de Data)

Algumas colunas numéricas foram corrompidas pelo Excel e interpretadas como datas. São elas: `IEG_2020`, `IPS_2020`, `IPP_2020`, `IAA_2021`, `IEG_2021`, `IPS_2021`, `IDA_2021`, `IPP_2021`, `IPV_2021`, `IPS_2022`. Essas colunas serão **descartadas** — o dataset 2024 tem as equivalentes limpas.

In [ ]:
DATETIME_CORRUPT_COLS = ['IEG_2020','IPS_2020','IPP_2020','IAA_2021','IEG_2021',
                          'IPS_2021','IDA_2021','IPP_2021','IPV_2021','IPS_2022']

print('Colunas com artefato de data (serão descartadas do modelo):')
for col in DATETIME_CORRUPT_COLS:
    if col in df_main.columns:
        sample = df_main[col].dropna().iloc[0] if len(df_main[col].dropna()) > 0 else 'vazio'
        print(f'  {col}: {sample}')

## 5. Análise das Features Numéricas (Dataset 2024 — base limpa)

In [ ]:
# Dataset 2024 é limpo — usamos para análise das features principais
features_2024 = ['INDE 22', 'IAA', 'IEG', 'IDA', 'IPV', 'IAN', 'Matem', 'Portug']
df_feat = df_2024[features_2024 + ['TARGET']].dropna()

print(f'Registros completos no dataset 2024: {len(df_feat)}')
df_feat[features_2024].describe().round(2)

In [ ]:
# Correlação com TARGET
corr = df_feat[features_2024].corrwith(df_feat['TARGET']).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in corr.values]
ax.barh(corr.index, corr.values, color=colors)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title('Correlação das Features com TARGET (Defasagem Binária)', fontsize=12)
ax.set_xlabel('Correlação de Pearson')
plt.tight_layout()
plt.show()

print('\nCorrelações:')
print(corr.round(3))

In [ ]:
# Distribuição das features por classe
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for i, feat in enumerate(features_2024):
    for label, color in [(0, '#2ecc71'), (1, '#e74c3c')]:
        data = df_feat[df_feat['TARGET'] == label][feat]
        axes[i].hist(data, bins=20, alpha=0.6, color=color,
                     label='No Nível' if label == 0 else 'Defasado')
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)

plt.suptitle('Distribuição das Features por Classe', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Análise das Features Categóricas

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# PEDRA_2022
pedra_order = ['Quartzo', 'Ágata', 'Ametista', 'Topázio']
pedra_counts = df_main.groupby('PEDRA_2022')['TARGET'].mean() * 100
pedra_counts = pedra_counts.reindex([p for p in pedra_order if p in pedra_counts.index])
axes[0].bar(pedra_counts.index, pedra_counts.values, color=['#e74c3c','#e67e22','#f1c40f','#2ecc71'])
axes[0].set_title('% Defasados por Pedra (2022)')
axes[0].set_ylabel('% Defasados')
axes[0].set_ylim(0, 100)

# PONTO_VIRADA_2022
pv = df_main.groupby('PONTO_VIRADA_2022')['TARGET'].mean() * 100
axes[1].bar(pv.index, pv.values, color=['#e74c3c','#2ecc71'])
axes[1].set_title('% Defasados por Ponto de Virada (2022)')
axes[1].set_ylabel('% Defasados')

# FASE_2022
fase = df_main.groupby('FASE_2022')['TARGET'].mean() * 100
axes[2].bar(fase.index.astype(str), fase.values, color='steelblue')
axes[2].set_title('% Defasados por Fase (2022)')
axes[2].set_ylabel('% Defasados')

plt.tight_layout()
plt.show()

## 7. Matriz de Correlação (Dataset 2024)

In [ ]:
corr_matrix = df_feat[features_2024 + ['TARGET']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Matriz de Correlação — Features + Target', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Conclusões e Decisões de Modelagem

### Features selecionadas para o modelo
Com base na análise, utilizaremos as seguintes features do dataset 2024 (mais limpo e consistente):

| Feature | Descrição | Correlação com Target |
|---|---|---|
| IAN | Indicador de Adequação ao Nível | Alta (negativa) |
| INDE 22 | Índice de Desenvolvimento Educacional | Alta (negativa) |
| IDA | Indicador de Aprendizagem | Moderada |
| IEG | Indicador de Engajamento | Moderada |
| IPV | Indicador de Ponto de Virada | Moderada |
| IAA | Indicador de Auto Avaliação | Baixa |
| Matem | Nota de Matemática | Moderada |
| Portug | Nota de Português | Moderada |

### Estratégia do modelo
- **Tipo:** Classificação binária (defasado = 1, não defasado = 0)
- **Algoritmo base:** RandomForestClassifier (robusto, sem necessidade de normalização)
- **Métrica principal:** F1-Score + AUC-ROC (dados com desbalanceamento moderado ~60/40)
- **Dados de treino:** Dataset 2024 (limpeza zero de formato, 860 registros)
- **Dados do principal:** Usados para features históricas adicionais se necessário

### Problemas identificados a tratar no preprocessing
1. Colunas corrompidas como datas no Excel (10 colunas) → descartar ou corrigir
2. `NOTA_ING_2022`: 79% nulos → excluir
3. Colunas object que devem ser float → conversão explícita
4. Padronizar nomes das colunas (dataset 2024 tem nomes diferentes do principal)

In [ ]:
# Resumo final: features selecionadas e dataset de treino
FEATURES_MODELO = ['IAN', 'INDE 22', 'IDA', 'IEG', 'IPV', 'IAA', 'Matem', 'Portug']
TARGET_COL = 'TARGET'

df_modelo = df_2024[FEATURES_MODELO].copy()
df_modelo[TARGET_COL] = (df_2024['Defas'] < 0).astype(int)
df_modelo = df_modelo.dropna(subset=[f for f in FEATURES_MODELO if f not in ['Portug','Matem']])

# Imputar notas com mediana
df_modelo['Matem'] = df_modelo['Matem'].fillna(df_modelo['Matem'].median())
df_modelo['Portug'] = df_modelo['Portug'].fillna(df_modelo['Portug'].median())

print(f'Dataset final para modelagem: {df_modelo.shape}')
print(f'Defasados (1): {df_modelo[TARGET_COL].sum()} ({df_modelo[TARGET_COL].mean()*100:.1f}%)')
print(f'Não defasados (0): {(df_modelo[TARGET_COL]==0).sum()} ({(df_modelo[TARGET_COL]==0).mean()*100:.1f}%)')
print(f'\nFe atures sem nulos: {df_modelo[FEATURES_MODELO].isnull().sum().sum()}')
print('\n✅ Pronto para iniciar a pipeline de treinamento!')